# **Import des modules nécessaires**

In [26]:
import pandas as pd #pour la manipulation de données
from pathlib import Path #pour la gestion des chemins de fichiers
import spacy #pour le prétraitement de texte
from spacy.lang.fr.stop_words import STOP_WORDS as spacy_stopwords

from bertopic import BERTopic #pour la modélisation de sujets

from umap import UMAP #pour la réduction de dimensionnalité
from hdbscan import HDBSCAN #pour le clustering de BERTopic
from sklearn.cluster import KMeans #pour le clustering de BERTopic

from sklearn.feature_extraction.text import CountVectorizer #pour la vectorisation de texte
from bertopic.vectorizers import ClassTfidfTransformer #pour la vectorisation de texte spécifique à BERTopic

from sentence_transformers import SentenceTransformer #pour les embeddings de phrases

# **Chargement du corpus de Zola et de spacy**


In [27]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "02_corpus_zola.csv", encoding="utf-8",)
df.head()


,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,La joie de vivre.,1884,1,1,Comme six heures sonnaient au coucou de la sal...,265
1,La joie de vivre.,1884,1,2,"Il ajouta, après une hésitation: Tu devrais al...",156
2,La joie de vivre.,1884,1,3,en voilà une morveuse qui peut se flatter de n...,265
3,La joie de vivre.,1884,1,4,Et on s’était à peine rencontré deux ou trois ...,255
4,La joie de vivre.,1884,1,5,Quelques gouttes de pluie volant dans l’ouraga...,188


In [28]:
df.shape

(653, 6)

# **Traitement du Corpus de Zola**

In [29]:
stop_perso = {
    "grand", "petit", "homme", "femme", "jour", "heure", "coup", "œil", "oeil", 
    "main", "bras", "tête", "voix", "milieu", "eau", "terre", "air", "monde", 
    "chose", "nuit", "vie", "enfant", "père", "mère", "fille", "garçon", 
    "monsieur", "madame", "falloir", "aller", "voir", "dire", "faire", 
    "pouvoir", "vouloir", "savoir", "venir", "devoir", "prendre", "donner",
    "oui", "non", "où", "quand", "comment", "bon", "jeune", "vieux", "suite"
}

In [30]:
# Chargement du modèle avec désactivation du 'parser' syntaxique pour gagner en vitesse
# On garde impérativement 'ner' pour repérer les personnages/lieux et 'lemmatizer'
nlp = spacy.load("fr_core_news_lg", disable=["parser"])
nlp.max_length = 2_000_000  

#on convertit en liste
textes_bruts = df["texte"].astype(str).tolist()

textes_nettoyes = []

# Utilisation de nlp.pipe pour traiter les textes par blocs (très rapide)
for doc in nlp.pipe(textes_bruts, batch_size=50): 
    tokens = [] # Liste pour stocker les tokens nettoyés
    
    for token in doc:
        lemme = token.lemma_.lower() # Obtenir le lemme du token en minuscules
        
        if (
            not token.is_stop # Ignorer les stop words spaCy par défaut
            and not token.is_punct # Ignorer la ponctuation
            and not token.like_num # Ignorer les chiffres
            and not token.is_space # Ignorer les espaces vides
            and token.ent_type_ not in ['PER', 'LOC', 'ORG'] # Ignorer les Personnages, Lieux et Organisations
            and token.pos_ in {"NOUN", "ADJ"}  # Garder Noms, Adjectifs ET Verbes
            and len(lemme) > 2 # Ignorer les mots de 1 ou 2 lettres
            and lemme not in stop_perso
        ):
            tokens.append(lemme)
            
    # Rejoindre les tokens validés et les ajouter à la liste finale
    textes_nettoyes.append(" ".join(tokens))

# Application de la liste nettoyée à la nouvelle colonne du DataFrame
df["phrases_lemm"] = textes_nettoyes

# Affichage du résultat
df[["phrases_lemm"]].head()

,phrases_lemm
0,coucou salle espoir fauteuil lourd jambe goutt...
1,hésitation coin route pâle colère rencontre pe...
2,morveux bourrique paisible accoutumé violence ...
3,peine fois douleur commerce voyage célébrité m...
4,goutte pluie volant ouragan visage souffle ter...


## **1) Choix du modèle d'embedding**

Ici je vais choisir un modèle d'embedding pré-entraîné pour transformer les textes en vecteurs numériques. Je vais utiliser un modèle de la bibliothèque Sentence Transformers, qui est compatible avec BERTopic.

In [31]:
embedding_model = SentenceTransformer(
    "dangvantuan/sentence-camembert-base"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [32]:
print("Génération des embeddings sémantiques...")
embeddings = embedding_model.encode(df['texte'].tolist(), show_progress_bar=True)

Génération des embeddings sémantiques...


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

## **2) Pipeline de Traitement**

### 3) CountVectorizer et ClassTfidfTransformer avec des stop words personnalisés 

In [38]:
hdbscan_model = HDBSCAN( min_cluster_size=10, 
                        min_samples=2, 
                        metric='euclidean', 
                        cluster_selection_method='eom',
                        prediction_data=True)

umap_model = UMAP( n_neighbors=20,
                  n_components=3, 
                  min_dist=0.0, 
                  metric="cosine",
                  random_state=42)


vectorizer_model = CountVectorizer(
    min_df=2,    # Le mot doit apparaître dans au moins 2 segments pour être pris en compte (élimine les fautes ou mots uniques)
    max_df=0.6)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
)


topic_model = BERTopic(
    language="french",
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=False,
    verbose=True,
    nr_topics="auto"
)
topics, probs = topic_model.fit_transform(df["phrases_lemm"].tolist(), embeddings= embeddings)

2026-07-08 14:51:12,561 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-08 14:51:13,295 - BERTopic - Dimensionality - Completed ✓
2026-07-08 14:51:13,295 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-08 14:51:13,302 - BERTopic - Cluster - Completed ✓
2026-07-08 14:51:13,302 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-07-08 14:51:13,316 - BERTopic - Representation - Completed ✓
2026-07-08 14:51:13,316 - BERTopic - Topic reduction - Reducing number of topics
2026-07-08 14:51:13,319 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-08 14:51:13,332 - BERTopic - Representation - Completed ✓
2026-07-08 14:51:13,332 - BERTopic - Topic reduction - Reduced number of topics from 24 to 21


In [39]:
new_topics = topic_model.reduce_outliers(
    df["phrases_lemm"].tolist(), 
    topics, 
    strategy="embeddings",
    embeddings=embeddings
)

# Met à jour le modèle avec ces nouveaux thèmes plus propres
topic_model.update_topics(df["phrases_lemm"].tolist(), topics=new_topics)

2026-07-08 14:51:15,941 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


## **3) Topics Présent**

In [40]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,0,100,0_lit_malade_visage_mal,"[lit, malade, visage, mal, chambre, chien, peu...",[curé sentencieux verre docteur souffrance com...
1,1,64,1_cœur_raison_parole_fois,"[cœur, raison, parole, fois, silence, docteur,...",[cruel misérable révolte sang mot abominable m...
2,2,49,2_chambre_lit_porte_silence,"[chambre, lit, porte, silence, table, docteur,...",[minute regard vaisselle table débandade chais...
3,3,38,3_vent_vague_mer_pêcheur,"[vent, vague, mer, pêcheur, galet, pluie, flot...",[misère tempête mai dernier maison falaise mar...
4,4,32,4_franc_idée_mer_fortune,"[franc, idée, mer, fortune, mariage, fils, aff...",[poli pièce bois abondance extraordinaire mot ...
5,5,40,5_ancien_soir_samedi_nouveau,"[ancien, soir, samedi, nouveau, chambre, ménag...",[ancien camarade pensée départ mois nouveau dé...
6,6,36,6_désir_amour_point_odeur,"[désir, amour, point, odeur, chair, joie, pass...",[malaise secousse profond maladie besoin conti...
7,7,43,7_chatte_maison_fois_table,"[chatte, maison, fois, table, patte, bête, jeu...",[planche planche superbe réalité saignant orga...
8,8,39,8_docteur_porte_sage_pauvre,"[docteur, porte, sage, pauvre, cuisine, temps,...",[drôle inutile pluie écart mine revêche bonjou...
9,9,29,9_mort_bougie_lit_sommeil,"[mort, bougie, lit, sommeil, soir, peur, chamb...",[ancien livre médecine peur dos unique sensati...


In [41]:
# Récupération de la dimension temporelle
timestamps = df['paquet_id'].tolist()

# Génération des topics dans le temps
topics_over_time = topic_model.topics_over_time(
    df['phrases_lemm'].tolist(), 
    timestamps, 
    nr_bins=30 
)

topic_model.visualize_topics_over_time(topics_over_time) #topics=themes_interet)

30it [00:00, 140.19it/s]


In [42]:
for topic_id in topic_info["Topic"].head(15):
    if topic_id != -1:
        print("\nTOPIC", topic_id)
        print(topic_model.get_topic(topic_id)[:15])


TOPIC 0
[('lit', np.float64(0.022202621179674115)), ('malade', np.float64(0.018594737666320933)), ('visage', np.float64(0.01836658485288974)), ('mal', np.float64(0.01742432468060245)), ('chambre', np.float64(0.017391135906795356)), ('chien', np.float64(0.017286197508602105)), ('peur', np.float64(0.016440939514285505)), ('face', np.float64(0.01588167103946921)), ('pauvre', np.float64(0.015761590475382056)), ('geste', np.float64(0.015400460524232891))]

TOPIC 1
[('cœur', np.float64(0.026993590916338066)), ('raison', np.float64(0.023393196096141627)), ('parole', np.float64(0.019148216743736523)), ('fois', np.float64(0.018137376052307357)), ('silence', np.float64(0.017456009662523485)), ('docteur', np.float64(0.0172870041727113)), ('mauvais', np.float64(0.017128091929831658)), ('mot', np.float64(0.017058967711189702)), ('sage', np.float64(0.016791585895939622)), ('tante', np.float64(0.015710408696271137))]

TOPIC 2
[('chambre', np.float64(0.03598417230557867)), ('lit', np.float64(0.022528